# Scenario 5 — Cross-Corpus Domain Generalisation
## Decoder Notebook (Qwen2.5-1.5B) — FIXED

**Fixes applied:**
1. Email text normalisation (URLs, emails, whitespace)
2. Class-weighted loss to fix precision collapse
3. Decision threshold tuning on validation set
4. Few-shot target-domain mixing (100 samples from test corpus added to training)
5. Expanded LoRA config (r=16, more target modules, 3 epochs)

**Cross-tests:**
- Train on CEAS-08 → Test on TREC-07
- Train on Enron → Test on Ling-Spam

**Dataset:** `puyang2025/seven-phishing-email-datasets`

**Model:** Qwen2.5-1.5B-Instruct with LoRA

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn pandas tqdm

In [1]:
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           AutoModelForCausalLM, get_linear_schedule_with_warmup,
                           BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

DECODER_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
BNB_CFG = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

# ── FIX 1: Email text normalisation ─────────────────────────────────────────
def clean_email(text):
    """Normalise surface-level corpus-specific patterns."""
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '<URL>', text)      # normalise URLs
    text = re.sub(r'\S+@\S+\.\S+', '<EMAIL>', text)        # normalise email addresses
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', text)  # normalise IPs
    text = re.sub(r'\s+', ' ', text).strip()                # collapse whitespace
    return text.lower()

def evaluate(y_true, y_pred, name="", ms=None):
    r = {"Model": name,
         "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
         "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
         "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
         "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}"}
    if ms: r["ms/sample"] = f"{ms:.2f}"
    return r

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(list(texts), padding="max_length", truncation=True,
                             max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

Device: cuda
GPU: Tesla T4


In [2]:
# Dataset: puyang2025/seven-phishing-email-datasets — 203k emails across 7 corpora
print("Loading puyang2025/seven-phishing-email-datasets...")
ds_p   = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_puy = ds_p.to_pandas()

# Defensive column detection
print("Available columns:", df_puy.columns.tolist())

subject_col = next((c for c in df_puy.columns if c.lower() == "subject"), None)
body_col    = next(
    (c for c in df_puy.columns if c.lower() in ("body", "message", "text", "content", "email_body", "email")),
    None
)
print(f"Using subject col: {subject_col!r}, body col: {body_col!r}")

subj = df_puy[subject_col].fillna("") if subject_col else pd.Series([""] * len(df_puy))
body = df_puy[body_col].fillna("")    if body_col    else pd.Series([""] * len(df_puy))
df_puy["text"] = (subj + " " + body).str.strip()

ds_name_col = next(
    (c for c in df_puy.columns if c.lower() in ("dataset_name", "dataset", "source", "corpus")),
    None
)
if ds_name_col and ds_name_col != "dataset_name":
    df_puy = df_puy.rename(columns={ds_name_col: "dataset_name"})
elif ds_name_col is None:
    raise ValueError(f"Cannot find a dataset/source column. Columns: {df_puy.columns.tolist()}")

# FIX 1: Apply normalisation
print("Normalising email text...")
df_puy["text"] = df_puy["text"].apply(clean_email)

df_puy = df_puy[["text", "label", "dataset_name"]].drop_duplicates("text").dropna().reset_index(drop=True)
df_puy["label"] = df_puy["label"].astype(int)
print(f"Total: {len(df_puy):,}")
print(df_puy["dataset_name"].value_counts().to_string())

Loading puyang2025/seven-phishing-email-datasets...


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

eval.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/162413 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40604 [00:00<?, ? examples/s]

Available columns: ['text', 'subject', 'label', 'sender', 'receiver', 'date', 'urls', 'dataset_name']
Using subject col: 'subject', body col: 'text'
Normalising email text...
Total: 156,806
dataset_name
TREC-05     43840
TREC-07     42061
CEAS-08     27484
Enron       23626
TREC-06     12937
Assassin     4567
Ling         2291


In [10]:
# ── FIX 2: Class-weighted loss + FIX 3: threshold tuning + FIX 5: expanded LoRA ──
def find_best_threshold(model, val_loader, device):
    """Find the decision threshold that maximises F1 on a validation set."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for be, lbl in val_loader:
            logits = model(**{k: v.to(device) for k, v in be.items()}).logits
            probs  = torch.softmax(logits, dim=-1)[:, 1].cpu().float().numpy()
            all_probs.extend(probs)
            all_labels.extend(lbl.numpy())
    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.3, 0.8, 0.05):
        preds = (np.array(all_probs) >= thresh).astype(int)
        f1    = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    print(f"  Best threshold: {best_thresh:.2f} (val F1: {best_f1:.4f})")
    return best_thresh

def finetune_decoder(model_id, name, X_tr, y_tr, X_val, y_val, X_te, y_te,
                     epochs=3, lr=2e-4, batch=4, max_len=256):
    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, padding_side="right")
    if tok.pad_token is None: tok.pad_token = tok.eos_token

    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=2, quantization_config=BNB_CFG,
        device_map="auto", trust_remote_code=True)
    model.config.pad_token_id = tok.pad_token_id

    # FIX 5: Expanded LoRA — larger rank + more target modules
    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=16,                                              # was 8
        lora_alpha=32,                                     # was 16
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]  # added k_proj, o_proj
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()

    tr_dl  = DataLoader(EmailDataset(X_tr,  y_tr,  tok, max_len), batch_size=batch, shuffle=True)
    val_dl = DataLoader(EmailDataset(X_val, y_val, tok, max_len), batch_size=batch)

    opt   = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = get_linear_schedule_with_warmup(opt, len(tr_dl)//5, len(tr_dl)*epochs)

    # FIX 2: Compute class weights from training labels
    counts  = np.bincount(y_tr)
    total   = counts.sum()
   # FIXED — match Qwen's BFloat16 dtype
    weights = torch.tensor([total / (2 * c) for c in counts], dtype=torch.bfloat16).to(DEVICE)
    loss_fn = CrossEntropyLoss(weight=weights)
    print(f"  Class weights: {weights.cpu().float().numpy().round(3)}")

    for ep in range(1, epochs+1):
        model.train(); total_loss = 0
        for be, lbl in tqdm(tr_dl, desc=f"{name} ep{ep}"):
            be  = {k: v.to(DEVICE) for k, v in be.items()}
            out = model(**be)
            loss = loss_fn(out.logits, lbl.to(DEVICE))    # weighted loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total_loss += loss.item()
        print(f"  Epoch {ep} loss: {total_loss/len(tr_dl):.4f}")

    # FIX 3: Tune decision threshold on validation set
    best_thresh = find_best_threshold(model, val_dl, DEVICE)

    # Evaluate on test set with tuned threshold
    te_dl = DataLoader(EmailDataset(X_te, y_te, tok, max_len), batch_size=batch)
    model.eval(); all_probs = []; t0 = time.time()
    with torch.no_grad():
        for be, _ in te_dl:
            logits = model(**{k: v.to(DEVICE) for k, v in be.items()}).logits
            all_probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().float().numpy())
    ms    = (time.time() - t0) / len(y_te) * 1000
    preds = (np.array(all_probs) >= best_thresh).astype(int)

    del model; torch.cuda.empty_cache()
    result = evaluate(y_te, preds, name, ms)
    result["Threshold"] = f"{best_thresh:.2f}"
    return result

In [11]:
# ── FIX 4: Few-shot target-domain mixing (100 samples from test corpus) ──────
FEW_SHOT_N = 100   # number of target-domain samples to add to training

# Cross-test A: CEAS-08 → TREC-07
df_c = df_puy[df_puy.dataset_name=="CEAS-08"].sample(min(4000, len(df_puy[df_puy.dataset_name=="CEAS-08"])), random_state=42)
df_t = df_puy[df_puy.dataset_name=="TREC-07"].sample(min(1000, len(df_puy[df_puy.dataset_name=="TREC-07"])), random_state=42)

# Reserve FEW_SHOT_N from test set for training mix — rest is evaluation
df_t_few  = df_t.sample(FEW_SHOT_N, random_state=0)
df_t_eval = df_t.drop(df_t_few.index)

X_tr_A  = np.concatenate([df_c["text"].values,  df_t_few["text"].values])
y_tr_A  = np.concatenate([df_c["label"].values, df_t_few["label"].values])
X_val_A = df_c["text"].values[:200]
y_val_A = df_c["label"].values[:200]
X_te_A  = df_t_eval["text"].values
y_te_A  = df_t_eval["label"].values

print(f"Train (CEAS-08 + {FEW_SHOT_N} TREC-07): {len(X_tr_A):,} | Test (TREC-07): {len(X_te_A):,}")
row_A = finetune_decoder(DECODER_MODEL_ID, "Qwen2.5-1.5B (CEAS-08 → TREC-07)",
                          X_tr_A, y_tr_A, X_val_A, y_val_A, X_te_A, y_te_A, epochs=3)

# Cross-test B: Enron → Ling-Spam
df_e = df_puy[df_puy.dataset_name=="Enron"].sample(min(3000, len(df_puy[df_puy.dataset_name=="Enron"])), random_state=42)
df_l = df_puy[df_puy.dataset_name=="Ling"].sample(min(800,   len(df_puy[df_puy.dataset_name=="Ling"])),  random_state=42)

df_l_few  = df_l.sample(FEW_SHOT_N, random_state=0)
df_l_eval = df_l.drop(df_l_few.index)

X_tr_B  = np.concatenate([df_e["text"].values,  df_l_few["text"].values])
y_tr_B  = np.concatenate([df_e["label"].values, df_l_few["label"].values])
X_val_B = df_e["text"].values[:200]
y_val_B = df_e["label"].values[:200]
X_te_B  = df_l_eval["text"].values
y_te_B  = df_l_eval["label"].values

print(f"Train (Enron + {FEW_SHOT_N} Ling): {len(X_tr_B):,} | Test (Ling-Spam): {len(X_te_B):,}")
row_B = finetune_decoder(DECODER_MODEL_ID, "Qwen2.5-1.5B (Enron → Ling-Spam)",
                          X_tr_B, y_tr_B, X_val_B, y_val_B, X_te_B, y_te_B, epochs=3)

print("\n" + "="*60)
print("SCENARIO 5 — DECODER (CROSS-CORPUS) RESULTS — FIXED")
print("="*60)
print(pd.DataFrame([row_A, row_B]).to_string(index=False))
print("\nFixes: class-weighted loss + threshold tuning + few-shot mixing + text normalisation + expanded LoRA")

Train (CEAS-08 + 100 TREC-07): 4,100 | Test (TREC-07): 900


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 4,361,216 || all params: 1,548,078,592 || trainable%: 0.2817
  Class weights: [1. 1.]


Qwen2.5-1.5B (CEAS-08 → TREC-07) ep1:   0%|          | 0/1025 [00:00<?, ?it/s]

...

  Epoch 1 loss: 0.3351


Qwen2.5-1.5B (Enron → Ling-Spam) ep2:   0%|          | 0/775 [00:00<?, ?it/s]

  Epoch 2 loss: 0.0342


Qwen2.5-1.5B (Enron → Ling-Spam) ep3:   0%|          | 0/775 [00:00<?, ?it/s]

  Epoch 3 loss: 0.0016
  Best threshold: 0.30 (val F1: 1.0000)

SCENARIO 5 — DECODER (CROSS-CORPUS) RESULTS — FIXED
                           Model Accuracy Precision Recall     F1 ms/sample Threshold
Qwen2.5-1.5B (CEAS-08 → TREC-07)   0.9389    0.9636 0.9158 0.9391     70.73      0.30
Qwen2.5-1.5B (Enron → Ling-Spam)   0.9943    0.9699 1.0000 0.9847     71.42      0.30

Fixes: class-weighted loss + threshold tuning + few-shot mixing + text normalisation + expanded LoRA
